[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/langchain-ai/langchain-academy/blob/main/module-2/chatbot-external-memory.ipynb) [![Open in LangChain Academy](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66e9eba12c7b7688aa3dbb5e_LCA-badge-green.svg)](https://academy.langchain.com/courses/take/intro-to-langgraph/lessons/58239440-lesson-6-chatbot-w-summarizing-messages-and-external-memory)

# Chatbot que resume la conversación y usa una base de datos como memoria externa

## Repaso

Vimos cómo personalizar el esquema del estado y el reducer del grafo.

También vimos varios trucos para recortar o filtrar mensajes en el estado del grafo.

Usamos estos conceptos en un chatbot con memoria que produce un resumen continuo de la conversación.

## Objetivos

Pero ¿y si queremos que nuestro chatbot tenga una memoria que persista indefinidamente?

Ahora presentaremos algunos checkpointers más avanzados que admiten bases de datos externas.

Aquí mostraremos cómo usar [Sqlite como checkpointer](https://docs.langchain.com/oss/python/langgraph/persistence#checkpointer-libraries), pero hay otros checkpointers disponibles, como Postgres.

In [ ]:
%%capture --no-stderr
%pip install --quiet -U langgraph-checkpoint-sqlite langchain_core langgraph langchain_openai

In [ ]:
import os, getpass

def _definir_variable(variable: str):
    if not os.environ.get(variable):
        os.environ[variable] = getpass.getpass(f"{variable}: ")

_definir_variable("OPENAI_API_KEY")

## Sqlite

Un buen punto de partida es el [checkpointer SqliteSaver](https://docs.langchain.com/oss/python/langgraph/persistence#checkpointer-libraries).

Sqlite es una base de datos SQL [pequeña, rápida y muy popular](https://x.com/karpathy/status/1819490455664685297).

Si le damos `":memory:"`, crea una base de datos Sqlite en memoria.

In [ ]:
import sqlite3
# En memoria
conn = sqlite3.connect(":memory:", check_same_thread = False)

Pero si le damos la ruta de una base de datos, ¡creará una por nosotros!

In [ ]:
# Descarga el archivo si no existe y se conecta a la base de datos local
!mkdir -p state_db && [ ! -f state_db/example.db ] && wget -P state_db https://github.com/langchain-ai/langchain-academy/raw/main/module-2/state_db/example.db

ruta_bd = "state_db/example.db"
conn = sqlite3.connect(ruta_bd, check_same_thread=False)

In [ ]:
# Este es nuestro checkpointer
from langgraph.checkpoint.sqlite import SqliteSaver
memoria = SqliteSaver(conn)

Volvamos a definir nuestro chatbot.

In [ ]:
from typing_extensions import Literal
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, RemoveMessage

from langgraph.graph import END
from langgraph.graph import MessagesState


modelo = ChatOpenAI(model="gpt-4o",temperature=0)

class Estado(MessagesState):
    resumen: str

# Define la lógica para llamar al modelo
def llamar_modelo(estado: Estado):

    # Obtiene el resumen si existe
    resumen = estado.get("resumen", "")

    # Si hay un resumen, lo agregamos
    if resumen:

        # Agrega el resumen al mensaje de sistema
        mensaje_sistema = f"Resumen de la conversación anterior: {resumen}"

        # Agrega el resumen antes de los mensajes más recientes
        mensajes = [SystemMessage(content=mensaje_sistema)] + estado["messages"]

    else:
        mensajes = estado["messages"]

    respuesta = modelo.invoke(mensajes)
    return {"messages": respuesta}

def resumir_conversacion(estado: Estado):

    # Primero, obtenemos el resumen existente, si lo hay
    resumen = estado.get("resumen", "")

    # Creamos el prompt para resumir
    if resumen:

        # Ya existe un resumen
        mensaje_resumen = (
            f"Este es el resumen de la conversación hasta ahora: {resumen}\n\n"
            "Amplía el resumen teniendo en cuenta los nuevos mensajes de arriba:"
        )

    else:
        mensaje_resumen = "Crea un resumen de la conversación de arriba:"

    # Agrega el prompt a nuestro historial
    mensajes = estado["messages"] + [HumanMessage(content=mensaje_resumen)]
    respuesta = modelo.invoke(mensajes)

    # Borra todos los mensajes menos los 2 más recientes
    mensajes_a_borrar = [RemoveMessage(id=m.id) for m in estado["messages"][:-2]]
    return {"resumen": respuesta.content, "messages": mensajes_a_borrar}

# Decide si terminar o resumir la conversación
def debe_continuar(estado: Estado)-> Literal ["resumir_conversacion",END]:

    """Devuelve el siguiente nodo a ejecutar."""

    mensajes = estado["messages"]

    # Si hay más de seis mensajes, resumimos la conversación
    if len(mensajes) > 6:
        return "resumir_conversacion"

    # Si no, podemos terminar
    return END

Ahora solo tenemos que volver a compilar con nuestro checkpointer de sqlite.

In [ ]:
from IPython.display import Image, display
from langgraph.graph import StateGraph, START

# Define un nuevo grafo
flujo = StateGraph(Estado)
flujo.add_node("conversacion", llamar_modelo)
flujo.add_node(resumir_conversacion)

# Define la conversación como punto de entrada
flujo.add_edge(START, "conversacion")
flujo.add_conditional_edges("conversacion", debe_continuar)
flujo.add_edge("resumir_conversacion", END)

# Compila
grafo = flujo.compile(checkpointer=memoria)
display(Image(grafo.get_graph().draw_mermaid_png()))

Ahora podemos invocar el grafo varias veces. 

In [ ]:
# Crea un hilo
config = {"configurable": {"thread_id": "1"}}

# Inicia la conversación
mensaje_entrada = HumanMessage(content="¡Hola! Soy Lance")
salida = grafo.invoke({"messages": [mensaje_entrada]}, config)
for m in salida['messages'][-1:]:
    m.pretty_print()

mensaje_entrada = HumanMessage(content="¿Cómo me llamo?")
salida = grafo.invoke({"messages": [mensaje_entrada]}, config)
for m in salida['messages'][-1:]:
    m.pretty_print()

mensaje_entrada = HumanMessage(content="¡Me gustan los 49ers!")
salida = grafo.invoke({"messages": [mensaje_entrada]}, config)
for m in salida['messages'][-1:]:
    m.pretty_print()

Confirmemos que nuestro estado se guardó localmente.

In [ ]:
config = {"configurable": {"thread_id": "1"}}
estado_grafo = grafo.get_state(config)
estado_grafo

### Persistir el estado

Usar una base de datos como Sqlite significa que el estado se persiste.

Por ejemplo, podemos reiniciar el kernel del notebook y comprobar que aún podemos cargar el estado desde la base de datos Sqlite que está en el disco.

In [ ]:
# Crea un hilo
config = {"configurable": {"thread_id": "1"}}
estado_grafo = grafo.get_state(config)
estado_grafo

## Studio

**⚠️ Aviso**

Desde que se filmaron estos videos, actualizamos Studio para que ahora pueda ejecutarse localmente y se acceda a él desde tu navegador. Esta es la forma preferida de ejecutar Studio, en lugar de la aplicación de escritorio que se muestra en el video. Ahora se llama _LangSmith Studio_ en lugar de _LangGraph Studio_. Encontrarás las instrucciones detalladas de configuración en la guía "Getting Setup" que está al inicio del curso. Puedes ver una descripción de Studio [aquí](https://docs.langchain.com/langsmith/studio), y los detalles del despliegue local [aquí](https://docs.langchain.com/langsmith/quick-start-studio#local-development-server).
Para iniciar el servidor de desarrollo local, ejecuta el siguiente comando en tu terminal, dentro del directorio `/studio` de este módulo:

```
langgraph dev
```

Deberías ver la siguiente salida:
```
- 🚀 API: http://127.0.0.1:2024
- 🎨 Studio UI: https://smith.langchain.com/studio/?baseUrl=http://127.0.0.1:2024
- 📚 API Docs: http://127.0.0.1:2024/docs
```

Abre tu navegador y entra a la URL de **Studio UI** que aparece arriba.
Carga el `chatbot` en Studio, que usa `module-2/studio/chatbot.py`, definido en `module-2/studio/langgraph.json`.